<a href="https://colab.research.google.com/github/Penugondanarasimha/AI_assisted_page_detector_py/blob/main/AI_assisted_page_detector_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import pandas as pd
import numpy as np
from google.colab import files

# If file doesn't exist in workspace, prompt an upload box automatically
if not os.path.exists("Volopay_SEO_Candidate_Data_Pack (1).xlsx"):
    print("Please upload your Excel file using the prompt below:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
else:
    filename = "Volopay_SEO_Candidate_Data_Pack (1).xlsx"

print(f"Using file: {filename}")

# Load the required workbook sheets
content = pd.read_excel(filename, sheet_name='Content', header=3)
ga4 = pd.read_excel(filename, sheet_name='GA4', header=3)
crm = pd.read_excel(filename, sheet_name='CRM', header=3)
crawl = pd.read_excel(filename, sheet_name='Crawl', header=3)
backlinks = pd.read_excel(filename, sheet_name='Backlinks', header=3)

# Aggregate GA4 metrics by landing page
ga4_p = ga4.groupby('landing_page').agg({
    'sessions': 'sum',
    'engaged_sessions': 'sum',
    'form_submits': 'sum'
}).reset_index()

# Aggregate CRM metrics by first landing page
crm_p = crm.groupby('first_landing_page').agg({
    'lead_id': 'count',
    'pipeline_value_usd': 'sum'
}).reset_index().rename(columns={'lead_id': 'crm_leads'})

# Merge multi-sheet data sources on URL key
df = content.merge(ga4_p, left_on='page', right_on='landing_page', how='left')
df = df.merge(crm_p, left_on='page', right_on='first_landing_page', how='left')
df = df.merge(crawl, left_on='page', right_on='url', how='left', suffixes=('', '_crawl'))
df = df.merge(backlinks, left_on='page', right_on='page', how='left', suffixes=('', '_backlinks'))

# Handle missing values safely
df.fillna({
    'sessions_90d': 0,
    'sessions': 0,
    'form_submits': 0,
    'crm_leads': 0,
    'pipeline_value_usd_x': 0,
    'pipeline_value_usd_y': 0,
    'internal_inlinks': 0,
    'page_ref_domains': 0
}, inplace=True)

df['pipeline_usd'] = df['pipeline_value_usd_x'] + df['pipeline_value_usd_y']

# Normalization & Underperformance Score (UPS) Calculation
max_sessions = df['sessions_90d'].max()
max_pipeline = df['pipeline_usd'].max()

df['traffic_score'] = df['sessions_90d'] / max_sessions if max_sessions > 0 else 0
df['pipeline_score'] = df['pipeline_usd'] / max_pipeline if max_pipeline > 0 else 0

# Weights: w1 = pipeline deficit, w2 = conversion rate deficit, w3 = technical/inlink penalty
w1, w2, w3 = 0.40, 0.35, 0.25
df['tech_penalty'] = df['internal_inlinks'].apply(lambda x: 1.0 if x < 5 else (0.5 if x < 15 else 0.0))

df['ups'] = 100 * (
    w1 * (1 - df['pipeline_score']) +
    w2 * (1 - (df['form_submits'] / df['sessions_90d'].replace(0, 1))) +
    w3 * df['tech_penalty']
)

# Rank pages by urgency score and output top results
ranked = df.sort_values(by='ups', ascending=False).reset_index(drop=True)
print(ranked[['page', 'sessions_90d', 'form_submits', 'pipeline_usd', 'internal_inlinks', 'ups']].head(8).to_string())

Using file: Volopay_SEO_Candidate_Data_Pack (1).xlsx
                                            page  sessions_90d  form_submits  pipeline_usd  internal_inlinks        ups
0  /blog/corporate-card-vs-business-credit-card/           980           0.0           0.0                12  87.500000
1                        /business-credit-cards/          1410          30.0       18000.0                 5  85.391683
2                             /spend-management/           980          24.0       24000.0                11  84.824675
3                                /virtual-cards/          1260          28.0       36000.0                13  83.994949
4                /blog/what-is-a-corporate-card/          8820          12.0       18000.0                61  73.588745
5          /blog/corporate-card-policy-template/          7340           1.0       60000.0                54  70.449777
6                           /solutions/startups/           930           0.0       72000.0                1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')